In [1]:
import pandas as pd
from scipy.stats import gmean

In [2]:
base = pd.read_csv('../dataset/dados historicos bitcoin.csv')

In [9]:
pd.set_option('display.max_rows', 10)

In [10]:
base

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
0,01.01.2026,"92.889,5","87.614,1","97.838,4","87.520,6","939,26K","6,02%"
1,01.12.2025,"87.612,7","90.372,2","94.591,4","83.858,1","1,62M","-3,06%"
2,01.11.2025,"90.374,2","109.602,7","111.229,8","80.697,7","2,34M","-17,54%"
3,01.10.2025,"109.602,8","114.048,7","126.186,0","103.632,7","2,24M","-3,90%"
4,01.09.2025,"114.048,5","108.247,3","117.910,4","107.274,7","1,34M","5,38%"
...,...,...,...,...,...,...,...
181,01.12.2010,"0,3","0,2","0,3","0,2","263,65K","44,09%"
182,01.11.2010,"0,2","0,2","0,5","0,1","826,25K","0,00%"
183,01.10.2010,"0,2","0,1","0,2","0,0","1,11M","210,99%"
184,01.09.2010,"0,1","0,1","0,2","0,1","216,81K","0,00%"


In [ ]:
base.dtypes

## 1. Limpeza e formatação dos dados

### Limpeza de strings e modificando valores

In [ ]:
base['Var%'] = base['Var%'].str.replace(',', '.')
base['Var%'] = base['Var%'].str.replace('%','')
base['Último'] = base['Último'].str.replace('.', '')
base['Último'] = base['Último'].str.replace(',', '.')
base['Abertura'] = base['Abertura'].str.replace('.', '')
base['Abertura'] = base['Abertura'].str.replace(',', '.')
base['Máxima'] = base['Máxima'].str.replace('.','')
base['Máxima'] = base['Máxima'].str.replace(',', '.')

In [ ]:
base['Abertura']

### Convertendo tipos

In [ ]:
base['Data'] = pd.to_datetime(base['Data'], format='%d.%m.%Y')

base['Último'] = pd.to_numeric(base['Último'])
base['Abertura'] = pd.to_numeric(base['Abertura'])

base['Var%'] = base['Var%'].astype(float)

In [ ]:
base.dtypes

### Criação de colunas

In [ ]:
base['Mês'] = base['Data'].dt.month
base['Ano'] = base['Data'].dt.year

In [ ]:
base[base['Mês'] == 8]

In [ ]:
base[base['Mês'] == 2]

## 2. Análise exploratória

In [ ]:
base_sem2026 = base[base['Ano'] != 2026]

base_filter = base_sem2026[['Data','Abertura', 'Último', 'Vol.', 'Var%', 'Mês', 'Ano']]

# Dataframe apenas com valores negativos
base_negativos = base_filter[base_filter['Var%'] <= 0]
tabela_final_ne = base_negativos.sort_values(['Mês', 'Ano'])

# Dataframe apenas com valores positivos
base_positivos = base_filter[base_filter['Var%'] > 0]
tabela_final_po = base_positivos.sort_values(['Mês', 'Ano'])

### Criação da lista de meses com mais variações negativas

In [ ]:
meses_negativos_7_ocorrencias = list()

print('MESES NEGATIVOS!')

for mes_n, tabela_n in tabela_final_ne.groupby('Mês'):
    if tabela_n['Var%'].count() >= 7:
        print(f"\n Foram encontrados {tabela_n['Var%'].count()} válores negativos no mês {mes_n} entre 2010 e 2025") # é apenas um log
        meses_negativos_7_ocorrencias.append(mes_n)
        

# Criação da lista de meses com mais variações negativas
meses_positivos_7_ocorrencias = list()

print("\n MESES POSITIVOS!")

for mes_p, tabela_p in tabela_final_po.groupby('Mês'):
    if tabela_p['Var%'].count() > 7:
        meses_positivos_7_ocorrencias.append(mes_p)
        print(f"\n Foram encontrados {tabela_p['Var%'].count()} válores positivos no mês {mes_p} entre 2010 e 2025") # é apenas um log

In [ ]:
meses_positivos_7_ocorrencias

### Cálculo de média negativos

In [ ]:
base_filter[base_filter['Mês'].isin(meses_negativos_7_ocorrencias)].groupby('Mês')['Var%'].mean()


### Cálculo de média positivos

In [ ]:
# base_filter[base_filter['Mês'].isin(meses_positivos_7_ocorrencias)].groupby('Mês')['Var%'].mean()
clean_data = base_filter['Var%'].dropna()

gmean(clean_data /100 + 1) -1

# OBS: todo número negativo é tratado como "nan", temos que tratar isso de alguma forma!

In [ ]:
"""
janeiro, junho e julho são meses com média de desempenho pior do que março e dezembro mesmo que março e dezembro tenha uma quantidade
de fechamentos negativos maior, por quê?
"""

In [ ]:
base.sort_values(by='Data')

In [ ]:
base_filter[base_filter['Mês'] == 8]

## Quantas vezes cada mês com piores desempenhos foram positivos?

In [ ]:
for mes_posi, tabela_p in base_positivos.groupby('Mês'):
    for valor in meses_negativos_7_ocorrencias:
        if mes_posi == valor:
            print("\n"f" Foram contabilizados {tabela_p['Var%'].count()} valores positivos no mês {mes_p}", "\n")

In [ ]:
for m, tabela_posi in base_positivos.groupby('Mês'):
    print(f" Foram contabilizados {tabela_posi['Var%'].count()} valores positivos no mês {m}", "\n")

## Quão grande é a queda em agosto e setembro comparado a queda em outros meses? devemos nos preocuparmos?

In [ ]:
A queda em agosto em média não costuma ser grande comparado a outros meses com mal desempenho históricamente,
tendo uma variação de apenas −0,66% a −2,26%.
A menos que seja ano de bearmarket, não há motivos para se preocupar com a queda em agosto visto que costuma 
ser uma correção saúdavel. O histórico mostra que em bearmakets, o preço pode alcançar patamares de até -38% de 
queda.

In [ ]:
tabela_media = tabela_final.copy()

In [ ]:
tabela_agosto = tabela_media[tabela_media['Mês'].isin(lista_meses)]

In [ ]:

for linha in meses_negativos_7_ocorrencias:
    tabela_para_media = tabela_agosto[tabela_agosto['Mês'] == linha]
    print(f"Mês {linha}: {tabela_para_media['Var%'].mean()}\n")
    print()

In [ ]:
base_filter_copia = base_filter.copy()
valores_negativos = base_negativos.copy()

In [ ]:
print(f"São {base_positivos['Var%'].count()} valores positivos na coluna 'Var%'")
print(f"São {base_negativos['Var%'].count()} valores negativos na coluna 'Var%'")

In [ ]:
base.to_excel("Dataset Bitcoin Formatado.xlsx")